In [3]:
## %pip install mujoco

  Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 65.7 MB/s eta 0:00:00a 0:00:01
Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [mujoco]2m7/8 [mujoco]]
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import Standard libraries
from pathlib import Path
import time

#Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer

In [27]:
# Settings
MJCF_PATH = Path("unitree_go2/scene_indoor.xml")

CONTROL_STEP = 0.002   # seconds per control update
PRINT_EVERY = 50

if not MJCF_PATH.is_file():
    raise FileNotFoundError(f"Could not find: {MJCF_PATH.resolve()}")

model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))
data = mujoco.MjData(model)

print("Loaded:", MJCF_PATH.resolve())
print("Actuators:", model.nu)
print("Sensors:", model.nsensor)

Loaded: /mnt/c/users/Raged Rhombus/Desktop/Projects/Go2-Learning/unitree_go2/scene_indoor.xml
Actuators: 12
Sensors: 30


In [22]:
## help(mujoco.mjtObj) ## To show the structure of the "object" of our model

In [10]:
ACTUATOR_NAMES = [
    mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    for i in range(model.nu)
]

SENSOR_NAMES = [
    mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_SENSOR, i)
    for i in range(model.nsensor)
]

print("Actuators:")
for i, name in enumerate(ACTUATOR_NAMES):
    print(i, name)

print("\nSensors:")
for i, name in enumerate(SENSOR_NAMES):
    print(i, name)

Actuators:
0 FL_hip
1 FL_thigh
2 FL_calf
3 FR_hip
4 FR_thigh
5 FR_calf
6 RL_hip
7 RL_thigh
8 RL_calf
9 RR_hip
10 RR_thigh
11 RR_calf

Sensors:
0 abduction_front_left_pos
1 hip_front_left_pos
2 knee_front_left_pos
3 abduction_hind_left_pos
4 hip_hind_left_pos
5 knee_hind_left_pos
6 abduction_front_right_pos
7 hip_front_right_pos
8 knee_front_right_pos
9 abduction_hind_right_pos
10 hip_hind_right_pos
11 knee_hind_right_pos
12 abduction_front_left_vel
13 hip_front_left_vel
14 knee_front_left_vel
15 abduction_hind_left_vel
16 hip_hind_left_vel
17 knee_hind_left_vel
18 abduction_front_right_vel
19 hip_front_right_vel
20 knee_front_right_vel
21 abduction_hind_right_vel
22 hip_hind_right_vel
23 knee_hind_right_vel
24 gyro
25 accelerometer
26 orientation
27 global_position
28 global_linvel
29 global_angvel


In [14]:
# control limits defined by the model

for i, name in enumerate(ACTUATOR_NAMES):
    print(
        name,
        "limited =", bool(model.actuator_ctrllimited[i]),
        "range =", model.actuator_ctrlrange[i]
    )

FL_hip limited = True range = [-0.9472  0.9472]
FL_thigh limited = True range = [-1.4  2.5]
FL_calf limited = True range = [-2.6227  -0.84776]
FR_hip limited = True range = [-0.9472  0.9472]
FR_thigh limited = True range = [-1.4  2.5]
FR_calf limited = True range = [-2.6227  -0.84776]
RL_hip limited = True range = [-0.9472  0.9472]
RL_thigh limited = True range = [-1.4  2.5]
RL_calf limited = True range = [-2.6227  -0.84776]
RR_hip limited = True range = [-0.9472  0.9472]
RR_thigh limited = True range = [-1.4  2.5]
RR_calf limited = True range = [-2.6227  -0.84776]


In [15]:
for i, name in enumerate(ACTUATOR_NAMES):
    print(
        name,
        "transmission:", model.actuator_trntype[i],
        "dynamics:", model.actuator_dyntype[i],
        "gain:", model.actuator_gainprm[i],
        "bias:", model.actuator_biasprm[i],
        "range:", model.actuator_ctrlrange[i],
    )

FL_hip transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-0.9472  0.9472]
FL_thigh transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-1.4  2.5]
FL_calf transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-2.6227  -0.84776]
FR_hip transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-0.9472  0.9472]
FR_thigh transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.    0.    0.    0.    0. ] range: [-1.4  2.5]
FR_calf transmission: 0 dynamics: 0 gain: [50.  0.  0.  0.  0.  0.  0.  0.  0.  0.] bias: [  0.  -50.   -0.5   0.    0.    0.

In [18]:
ACTUATOR_TO_POSITION_SENSOR = {
    "FL_hip":   "abduction_front_left_pos",
    "FL_thigh": "hip_front_left_pos",
    "FL_calf":  "knee_front_left_pos",

    "FR_hip":   "abduction_front_right_pos",
    "FR_thigh": "hip_front_right_pos",
    "FR_calf":  "knee_front_right_pos",

    "RL_hip":   "abduction_hind_left_pos",
    "RL_thigh": "hip_hind_left_pos",
    "RL_calf":  "knee_hind_left_pos",

    "RR_hip":   "abduction_hind_right_pos",
    "RR_thigh": "hip_hind_right_pos",
    "RR_calf":  "knee_hind_right_pos",
}

In [28]:
home_id = mujoco.mj_name2id(
    model, mujoco.mjtObj.mjOBJ_KEY, "home"
)

mujoco.mj_resetDataKeyframe(model, data, home_id)
mujoco.mj_forward(model, data)

In [29]:
standing_targets = data.ctrl.copy()
dt = model.opt.timestep
next_report = 1.0

with mujoco.viewer.launch_passive(model, data) as viewer:
    with viewer.lock():
        viewer.cam.distance = 1.5
        viewer.cam.azimuth = 135
        viewer.cam.elevation = -20
        viewer.cam.lookat[:] = data.body("base").xpos

    # Run for 10 simulated seconds, or until the window closes.
    while viewer.is_running() and data.time < 10.0:
        start = time.perf_counter()

        # Existing PD actuators track these 12 desired angles.
        data.ctrl[:] = standing_targets

        mujoco.mj_step(model, data)
        viewer.sync()

        if data.time >= next_report:
            height = float(data.body("base").xpos[2])
            speed = float(data.sensor("global_linvel").data[0])
            print(
                f"t={data.time:.1f}s | "
                f"height={height:.3f}m | "
                f"X speed={speed:.3f}m/s"
            )
            next_report += 1.0

        time.sleep(max(0.0, dt - (time.perf_counter() - start)))
    

t=1.0s | height=0.250m | X speed=-0.009m/s
t=2.0s | height=0.248m | X speed=-0.001m/s
t=3.0s | height=0.247m | X speed=-0.000m/s
t=4.0s | height=0.247m | X speed=-0.000m/s
t=5.0s | height=0.247m | X speed=-0.000m/s
t=6.0s | height=0.247m | X speed=-0.000m/s
t=7.0s | height=0.247m | X speed=-0.000m/s
t=8.0s | height=0.247m | X speed=-0.000m/s
t=9.0s | height=0.247m | X speed=-0.000m/s
t=10.0s | height=0.247m | X speed=-0.000m/s
